# WSP: baseline, tìm hướng gây lỗi, tự tìm bản sửa, kiểm trên dev (một notebook)

Model: `Ma7ee7/Qwen3.8_4B_Distilled` (revision `f33da883e0dadd9a4026556274e87489cd544d77`). Cài đặt: **GPU T4 x2**, **Internet On**.
Input: dataset riêng tư chứa `zaic_code_20261111_v5.zip` và `zaic_data_noaudio_20261107.zip`. Không cần output notebook nào khác.

Các bước:
1. Baseline trên dev (300 mẫu; QUICK: 10), think on.
2. Baseline trên practice (100 mẫu train, không thuộc dev; QUICK: 20), rồi tìm hướng gây lỗi (`dirs.pt`).
3-4. `auto_search.py` thử sửa trên practice, chỉ giữ khi qua luật cố định (DECISIONS l). Không bắt đầu vòng mới
   nếu vòng đó sẽ vượt `HOURS - FINAL_RESERVE_H` giờ.
5. Kiểm bản tốt nhất trên dev (cùng tập, cùng chế độ), in bảng so với baseline.

Nhãn: `bad` = thinking bị cắt hoặc lặp; `good` = điểm đúng 1,0; mẫu sai vì lý do khác không dùng để tìm hướng.
Chạy thử trước với `QUICK = True`, rồi `QUICK = False` và Save & Run All (Commit).
Kết quả: `/kaggle/working/wsp_results.zip` (ghi lại sau mỗi bước). Không gọi API ngoài; không chạy trên test BTC.

In [ ]:
# ===== CẤU HÌNH =====
QUICK = True          # True: chạy thử (10 mẫu dev, 20 mẫu practice) để đo tốc độ. False: chạy đầy đủ (Save & Run All)
MODE = "on"           # WSP nhắm vào thinking bị cắt/lặp nên dùng "on". Cờ "off" đã kiểm: không đổi đầu ra, không đo lại.
BUDGET = 12           # số vòng sửa tối đa (blog: 10, 25, 50). Hạn giờ bên dưới thường dừng trước
HOURS = 11.0          # tổng giờ cho phép dùng (Commit tối đa 12 giờ; chừa 1 giờ dự phòng)
FINAL_RESERVE_H = 3.5 # giờ chừa cho bước 5 (300 mẫu dev, ước 35 giây/mẫu, CHƯA XÁC MINH)

In [ ]:
# ĐƯỜNG DẪN INPUT (chỉ sửa nếu cần):
#  - CODE_ZIP: phải TRÙNG tên file zip mã đã upload vào dataset (hiện: zaic_code_20261111_v5.zip).
#  - Dữ liệu (zaic_data_noaudio_20261107.zip) và train/train.jsonl: không cần sửa, ô tự tìm dưới /kaggle/input/**.
#  - Nếu assert báo lỗi: kiểm tra dataset có đúng tên zip ở trên và là bản v5 (có --label-rule).
import glob, os, shutil, subprocess, sys, zipfile, json, time
NB_T0 = time.time()  # mốc bắt đầu notebook, để tính hạn giờ
W = "/kaggle/working"; T = "/tmp"
CODE_ZIP = "zaic_code_20261111_v5.zip"
for z in glob.glob("/kaggle/input/**/*.zip", recursive=True):   # trường hợp Kaggle không tự giải nén
    n = os.path.basename(z)
    if "zaic_code" in n:
        zipfile.ZipFile(z).extractall(f"{T}/code_src/{n[:-4]}")
    elif "zaic_data" in n:
        zipfile.ZipFile(z).extractall(f"{T}/data_root")
cands = sorted(glob.glob(f"{T}/code_src/**/optimize/wsp/auto_search.py", recursive=True)
               + glob.glob("/kaggle/input/**/optimize/wsp/auto_search.py", recursive=True))
cands = [p for p in cands if CODE_ZIP[:-4] in p] or cands   # ưu tiên đúng bản zip, tránh lấy nhầm bản cũ
assert cands, f"không thấy mã nguồn (optimize/wsp/auto_search.py): kiểm tra dataset có {CODE_ZIP}"
assert "--deadline-ts" in open(cands[0], encoding="utf-8").read(), f"mã nguồn cũ ({cands[0]}): cần {CODE_ZIP}"
REPO = W + "/ZAIC_2026"
shutil.copytree(os.path.dirname(os.path.dirname(os.path.dirname(cands[0]))), REPO, dirs_exist_ok=True)
tj = glob.glob(f"{T}/data_root/**/train/train.jsonl", recursive=True) or glob.glob("/kaggle/input/**/train/train.jsonl", recursive=True)
assert tj, "không thấy dữ liệu (train/train.jsonl)"
ROOT = os.path.dirname(os.path.dirname(tj[0]))
print("REPO =", REPO, "(từ", cands[0], ")"); print("ROOT =", ROOT, os.listdir(ROOT))

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!pip install -q vllm==0.33.0   # bản đã chạy được trên Kaggle T4 ngày 09/11
import torch, importlib.metadata as md
DTYPE = "auto" if torch.cuda.get_device_capability(0) >= (8, 0) else "half"   # T4 không có bf16
GPUS = ",".join(str(i) for i in range(torch.cuda.device_count()))
print("torch", torch.__version__, "| vllm", md.version("vllm"), "| dtype:", DTYPE, "| GPU:", GPUS)

In [ ]:
from huggingface_hub import snapshot_download
# cùng bản với máy cá nhân (docs/sources.md, DECISIONS 09/11); để ở /tmp cho khỏi bị lưu vào Output
MODEL_ID, MODEL_REV = "Ma7ee7/Qwen3.8_4B_Distilled", "f33da883e0dadd9a4026556274e87489cd544d77"
M = snapshot_download(MODEL_ID, revision=MODEL_REV, local_dir=T + "/model")
print(M, sorted(os.listdir(M)))

In [ ]:
def run(args):
    cmd = [sys.executable] + args
    print(">>", " ".join(cmd), flush=True)
    t0 = time.time()
    subprocess.run(cmd, cwd=REPO, check=True)
    print(f"   xong sau {time.time() - t0:.0f}s | đã chạy {(time.time() - NB_T0) / 3600:.2f} giờ", flush=True)

def head(src, n, dst):
    ids = [l.strip() for l in open(f"{REPO}/{src}", encoding="utf-8") if l.strip()][:n]
    open(dst, "w", encoding="utf-8").write("\n".join(ids) + "\n"); return dst

def summary(p):
    s = json.load(open(p, encoding="utf-8")); s.pop("per_sample", None); return s

def pack(name="wsp_results.zip"):  # gọi sau mỗi bước: Commit bị dừng giữa chừng vẫn còn kết quả các bước đã xong
    out = f"{W}/{name}"
    files = glob.glob(f"{W}/dev_*/*") + glob.glob(f"{W}/search/*") + \
            [f"{W}/{x}" for x in ("dirs.pt", "dirs.stats.json", "best_spec.json", "run_config.json")]
    with zipfile.ZipFile(out, "w", zipfile.ZIP_DEFLATED) as z:
        for f in files:
            if os.path.isfile(f):
                z.write(f, os.path.relpath(f, W))
    print("->", out, round(os.path.getsize(out) / 1e6, 2), "MB", flush=True)

DEV = head("eval/splits/dev_ids.txt", 10, W + "/dev_quick.txt") if QUICK else f"{REPO}/eval/splits/dev_ids.txt"
PRACTICE = head("optimize/wsp/practice100_20261107.txt", 20, W + "/practice_quick.txt") if QUICK else f"{REPO}/optimize/wsp/practice100_20261107.txt"
K = ["--base-model", M, "--root", ROOT, "--llm-dtype", DTYPE, "--gpu-mem", "0.9", "--gpus", GPUS, "--model-out", T + "/wsp_model"]
S = f"{W}/search"
print("QUICK:", QUICK, "| MODE:", MODE, "| DEV:", DEV, "| PRACTICE:", PRACTICE)

In [ ]:
# Bước 1: baseline trên dev, model gốc
json.dump({"quick": QUICK, "mode": MODE, "model": MODEL_ID, "revision": MODEL_REV, "code_zip": CODE_ZIP,
           "dev": os.path.basename(DEV), "practice": os.path.basename(PRACTICE)},
          open(W + "/run_config.json", "w"), indent=1)
run(["optimize/wsp/run_round.py", "--round", "0", "--spec", "none", "--think", MODE, "--ids", DEV, "--work", f"{W}/dev_{MODE}"] + K)
pack()
s = summary(f"{W}/dev_{MODE}/r00.summary.json")
print(f"== baseline dev think {MODE}: mean={s['mean_score']} by_kind={s['by_kind']} flags={s['flags']}")

In [ ]:
# Bước 2: baseline trên practice (= vòng 0 của bước tìm), rồi tìm hướng gây lỗi
run(["optimize/wsp/run_round.py", "--round", "0", "--spec", "none", "--think", MODE, "--ids", PRACTICE, "--work", S] + K)
exp = S + "/r00.export.jsonl"
labels = [json.loads(l)["label"] for l in open(exp, encoding="utf-8")]
NB, NG = labels.count("bad"), labels.count("good")
N = summary(S + "/r00.summary.json").get("n") or len([l for l in open(PRACTICE, encoding="utf-8") if l.strip()])
print(f"practice: bad {NB} | good {NG} | không dùng (sai vì lý do khác) {N - NB - NG} | flags {summary(S + '/r00.summary.json')['flags']}")
HAVE_DIRS = NB >= 2 and NG >= 2
if HAVE_DIRS:
    run(["optimize/wsp/directions.py", "--model", M, "--root", ROOT, "--export", exp, "--think", MODE,
         "--out", W + "/dirs.pt", "--stats", W + "/dirs.stats.json"])
    print(json.dumps(json.load(open(W + "/dirs.stats.json"))["layers"][:10], indent=0))
elif NB < 2:
    print(f"THIẾU MẪU BAD ({NB} < 2): ít thinking bị cắt/lặp ở chế độ '{MODE}'. WSP ở đây chỉ nhắm vào lỗi cắt/lặp, "
          "nên không có gì để gỡ. Bỏ qua bước sửa; quyết định go/no-go dựa trên bảng baseline.")
else:
    print(f"THIẾU MẪU GOOD ({NG} < 2): lỗi cắt/lặp nhiều ({NB} mẫu) nhưng quá ít mẫu đúng hoàn toàn (điểm 1,0) để so hướng. "
          "Lỗi có thật, chỉ thiếu mẫu đối chứng. Nếu đang QUICK: chạy bản đầy đủ (100 mẫu practice). Nếu bản đầy đủ vẫn thiếu: gửi kết quả cho Opus.")
pack()

In [ ]:
# Bước 3-4: tự tìm bản sửa trên practice (vòng 0 đã chạy ở bước 2, không chạy lại)
BUDGET_RUN = 2 if QUICK else BUDGET
DEADLINE = NB_T0 + (HOURS - FINAL_RESERVE_H) * 3600
if HAVE_DIRS:
    print(f"hạn bắt đầu vòng mới: {(DEADLINE - NB_T0) / 3600:.1f} giờ sau khi mở notebook | budget {BUDGET_RUN}")
    run(["optimize/wsp/auto_search.py", "--base-model", M, "--root", ROOT, "--dirs", W + "/dirs.pt", "--stats", W + "/dirs.stats.json",
         "--work", S, "--model-out", T + "/wsp_model", "--select-ids", PRACTICE, "--think", MODE, "--llm-dtype", DTYPE,
         "--gpus", GPUS, "--budget", str(BUDGET_RUN), "--deadline-ts", str(DEADLINE), "--round-sec-init", "3600"])
BEST = json.load(open(S + "/best_spec.json")) if os.path.exists(S + "/best_spec.json") else []
print("best_spec:", BEST)
pack()

In [ ]:
# Bước 5: kiểm bản tốt nhất trên dev (cùng tập, cùng chế độ với baseline). Không có bản sửa nào qua luật giữ thì bỏ qua.
if BEST:
    json.dump(BEST, open(W + "/best_spec.json", "w"), indent=1)
    run(["optimize/wsp/run_round.py", "--round", "1", "--spec", W + "/best_spec.json", "--dirs", W + "/dirs.pt",
         "--think", MODE, "--ids", DEV, "--work", f"{W}/dev_{MODE}"] + K)
rows = [json.loads(l) for l in open(f"{W}/dev_{MODE}/wsp_log.jsonl", encoding="utf-8")]
print(f"{'cấu hình':28s} {'n':>4s} {'mean':>7s} {'gen_tokens':>10s}  flags")
for r in rows:
    name = f"think {r['think']} " + ("WSP " + str([(o['layer'], o['top_k_cols'], o['alpha']) for o in r['spec']]) if r["spec"] else "gốc")
    print(f"{name[:28]:28s} {r['n']:4d} {r['mean_score']:7.4f} {str(r['gen_tokens_mean']):>10s}  {r['flags']}")
print("Go/no-go (plan bước 5): WSP phải cao hơn baseline gốc trên dev, gen_tokens không tăng. QUICK chỉ để đo tốc độ, không kết luận.")
print(f"Tổng thời gian: {(time.time() - NB_T0) / 3600:.2f} giờ")
pack()

## Vòng sửa tay (tuỳ chọn, sau khi Opus đọc kết quả trên máy)

Opus (chạy trên máy, không phải trên Kaggle) đọc `wsp_results.zip` và viết spec mới. Dán spec vào ô dưới, chạy.
Vòng tay đo trên practice (cùng log `search/`), không đo trên dev. Gửi lại dòng cuối `search/wsp_log.jsonl`.

In [ ]:
spec = []  # THAY bằng spec thật do Opus viết. Spec rỗng thì ô này bỏ qua (không báo lỗi, để Save & Run All vẫn hoàn tất)
if not spec:
    print("Bỏ qua vòng tay: chưa có spec. Dán spec vào `spec` rồi chạy lại ô này.")
elif not os.path.exists(W + "/dirs.pt"):
    print("Bỏ qua vòng tay: không có dirs.pt (không tìm được hướng).")
else:
    ROUND = 1 + max(json.loads(l)["round"] for l in open(S + "/wsp_log.jsonl", encoding="utf-8"))
    sp = f"{S}/r{ROUND:02d}.spec.json"; json.dump(spec, open(sp, "w"), indent=1)
    run(["optimize/wsp/run_round.py", "--round", str(ROUND), "--spec", sp, "--dirs", W + "/dirs.pt",
         "--think", MODE, "--ids", PRACTICE, "--work", S] + K)
    print(open(S + "/wsp_log.jsonl", encoding="utf-8").read().splitlines()[-1])
    pack()